# Gemma 3 1B running coach → LiteRT `.task` (for Google AI Edge Gallery)

**Run on Google Colab. Runtime → Change runtime type → T4 GPU.**
The adapter from Unsloth Desktop is in **MLX LoRA format**, so Cell 1 merges it manually (PEFT can't load it).
There are **two required runtime restarts** (protobuf conflicts): after 2a, and before Cell 3.
Files on disk (`merged_gemma3_1b/`, the `.tflite`) survive restarts, so you never redo earlier cells.
Follows Google's guide: https://ai.google.dev/gemma/docs/conversions/hf-to-mediapipe-task

In [ ]:
# [1] Upload adapter.zip, then merge the MLX-LoRA into base Gemma -> merged_gemma3_1b
!pip -q install transformers safetensors huggingface_hub torch
from google.colab import files
up = files.upload()                      # <- pick adapter.zip
import zipfile, glob, json, torch, shutil
zipfile.ZipFile(list(up)[0]).extractall('adapter_src')
ADAPTER = glob.glob('adapter_src/**/adapter_config.json', recursive=True)[0].rsplit('/',1)[0]
cfg = json.load(open(f'{ADAPTER}/adapter_config.json'))
BASE  = cfg.get('base_model_name_or_path','unsloth/gemma-3-1b-it')
lp    = cfg.get('lora_parameters', {})
scale = float(lp.get('scale', 2.0)); keys = lp.get('keys', [])
print('base=',BASE,'scale=',scale,'targets=',len(keys))
from transformers import AutoModelForCausalLM, AutoTokenizer
from safetensors import safe_open
from huggingface_hub import hf_hub_download
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.bfloat16)
sd = model.state_dict(); wf = glob.glob(f'{ADAPTER}/*.safetensors')[0]; merged = 0
with safe_open(wf, 'pt') as f:
    av = set(f.keys())
    for p in keys:
        if f'{p}.lora_a' in av and f'{p}.lora_b' in av and f'{p}.weight' in sd:
            a = f.get_tensor(f'{p}.lora_a').float(); b = f.get_tensor(f'{p}.lora_b').float()
            W = sd[f'{p}.weight']; W.add_(((a @ b).T * scale).to(W.dtype)); merged += 1
print('merged', merged, 'modules')
model.save_pretrained('merged_gemma3_1b', safe_serialization=True)
AutoTokenizer.from_pretrained(BASE).save_pretrained('merged_gemma3_1b')
shutil.copy(hf_hub_download(BASE, 'tokenizer.model'), 'merged_gemma3_1b/tokenizer.model')  # SentencePiece for the bundler
print('merged checkpoint ready -> merged_gemma3_1b')

In [ ]:
# [2a] Install the converter. Colab's protobuf (5.29) is too old for litert-torch (needs >=6.31).
# After this finishes: Runtime -> Restart session, then run cell [2b]. (merged_gemma3_1b/ persists.)
!pip -q install litert-torch 'protobuf==6.31.1'
print('installed -> now Runtime > Restart session, then run cell [2b]')

In [ ]:
# [2b] (after restart) Convert HF safetensors -> LiteRT .tflite. Takes ~10-30 min.
from litert_torch.generative.examples.gemma3 import gemma3
from litert_torch.generative.utilities import converter
from litert_torch.generative.utilities.export_config import ExportConfig
from litert_torch.generative.layers import kv_cache
m = gemma3.build_model_1b('merged_gemma3_1b')
ec = ExportConfig(); ec.kvcache_layout = kv_cache.KV_LAYOUT_TRANSPOSED; ec.mask_as_input = True
converter.convert_to_tflite(m, output_path='.', output_name_prefix='running-coach-gemma3-1b',
    prefill_seq_len=2048, kv_cache_max_len=4096, quantize='dynamic_int8', export_config=ec)
import glob; print('done:', glob.glob('*.tflite'))

In [ ]:
# [3] (Runtime -> Restart session FIRST, to clear the protobuf bump) Bundle .tflite + tokenizer -> .task
!pip -q install 'mediapipe==0.10.14'  # 0.10.21+/1.0.0 wheels dropped the genai bundler on py3.12
from mediapipe.tasks.python.genai import bundler
import glob
tflite = sorted(glob.glob('*.tflite'))[0]
cfg = bundler.BundleConfig(
    tflite_model=tflite, tokenizer_model='merged_gemma3_1b/tokenizer.model',
    start_token='<bos>', stop_tokens=['<eos>','<end_of_turn>'],
    output_filename='running-coach-gemma3-1b.task',
    prompt_prefix='<start_of_turn>user\n',
    prompt_suffix='<end_of_turn>\n<start_of_turn>model\n')
bundler.create_bundle(cfg)
print('wrote running-coach-gemma3-1b.task')
from google.colab import files; files.download('running-coach-gemma3-1b.task')